# VERA — CALVIN Ablation Training
**9 ablations × 3 seeds on CALVIN D→D**

### Before running:
1. `Runtime → Change runtime type → T4 GPU` (free) or A100 (Pro)
2. Run cells top to bottom — dataset downloads automatically inside Colab (first time only, saves to Drive)

### If session dies mid-run:
- Checkpoints are saved to Drive after every epoch — nothing is lost
- Re-run all setup cells (dataset skips re-download if already on Drive)
- In Cell 7, set `START_FROM` to the ablation index that was running when it died

In [1]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:  ', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → GPU')

GPU available: True
Device: Tesla T4
VRAM:   15.6 GB


In [2]:
# ── Cell 2: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints and results will be saved here — survives session crashes
DRIVE_ROOT = '/content/drive/MyDrive/VERA_CALVIN'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
print('Drive mounted. Output dir:', DRIVE_ROOT)

Mounted at /content/drive
Drive mounted. Output dir: /content/drive/MyDrive/VERA_CALVIN


In [3]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# torch / numpy / PIL already in Colab — only need CLIP and ftfy
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q ftfy regex

import clip, numpy, PIL, yaml
print('clip:', clip.__version__ if hasattr(clip,'__version__') else 'ok')
print('torch:', torch.__version__)
print('All dependencies ready')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
clip: ok
torch: 2.10.0+cu128
All dependencies ready


In [4]:
# ── Cell 4: Clone repo ───────────────────────────────────────────────────────
import os

REPO_URL  = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR  = '/content/RLConditionedVLA'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())
!ls

Cloning into '/content/RLConditionedVLA'...
remote: Enumerating objects: 175, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 175 (delta 12), reused 8 (delta 2), pack-reused 147 (from 1)
Receiving objects: 100% (175/175), 41.99 MiB | 22.27 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Working dir: /content/RLConditionedVLA
configs  envs	     README.md	     requirements.txt  VERA_CALVIN_Colab.ipynb
data	 evaluation  references.bib  scripts
docs	 models      REPORT.md	     training


In [6]:
import subprocess, os
subprocess.run(['pkill', '-f', 'aria2c'], capture_output=True)
partial = f'{DRIVE_ROOT}/task_D_D.zip'
if os.path.exists(partial): os.remove(partial)
print('Killed')

Killed


In [8]:
# ── Cell 5: CALVIN dataset — smart loader ────────────────────────────────────
import os, urllib.request, numpy as np, shutil
from pathlib import Path
from tqdm.auto import tqdm

CALVIN_BASE  = 'http://calvin.cs.uni-freiburg.de/dataset/task_D_D'
CALVIN_ZIP   = f'{DRIVE_ROOT}/task_D_D.zip'
CALVIN_PATH  = '/content/calvin_data/task_D_D'
DRIVE_CACHE  = f'{DRIVE_ROOT}/calvin_cache'
TRAIN_EP     = 1000
VAL_EP       = 200

# ── Step 1: Check what we already have ───────────────────────────────────────
zip_exists  = os.path.exists(CALVIN_ZIP)
zip_size_gb = os.path.getsize(CALVIN_ZIP) / 1e9 if zip_exists else 0
zip_complete = zip_size_gb > 160   # full zip is 165 GB

print(f'Zip on Drive:     {zip_exists}  ({zip_size_gb:.1f} GB)')
print(f'Zip complete:     {zip_complete}')

# Check if individual files are served by the server
print('\nTesting if server serves individual files...')
test_url = f'{CALVIN_BASE}/training/lang_annotations/auto_lang_ann.npy'
try:
    req = urllib.request.Request(test_url, method='HEAD')
    resp = urllib.request.urlopen(req, timeout=10)
    individual_files_ok = resp.status == 200
    print(f'Individual file access: YES (HTTP {resp.status})')
except Exception as e:
    individual_files_ok = False
    print(f'Individual file access: NO — {e}')

print()

# ── Step 2: Choose strategy ───────────────────────────────────────────────────
if zip_complete:
    print('STRATEGY: Selective extraction from complete zip on Drive')
    STRATEGY = 'extract_from_zip'
elif individual_files_ok:
    print('STRATEGY: Download individual NPZ files (~3-5 GB)')
    STRATEGY = 'individual_files'
else:
    print('STRATEGY: Must download full zip (server only serves zip)')
    print(f'\nZip already on Drive: {zip_size_gb:.1f} / 165 GB')
    print('Options:')
    print('  A) Resume zip download (need ~165 GB Drive space)')
    print('  B) Run !df -h to check Colab local disk space')
    print('     If >165 GB free, download zip directly to /content/ (not Drive)')
    STRATEGY = 'need_full_zip'

print(f'\nChosen strategy: {STRATEGY}')

# ── Step 3: Execute strategy ──────────────────────────────────────────────────

def load_ann(split):
    ann_path = f'{CALVIN_PATH}/{split}/lang_annotations/auto_lang_ann.npy'
    ann = np.load(ann_path, allow_pickle=True).item()
    return ann['info']['indx'], ann['language']['task']

def needed_indices(indx, max_ep):
    needed = set()
    for s, e in indx[:max_ep]:
        needed.update(range(s, e+1))
    return sorted(needed)

# ─── Strategy A: extract from complete zip ────────────────────────────────────
if STRATEGY == 'extract_from_zip':
    import zipfile
    print('Opening zip central directory (takes ~1 min for 165 GB zip)...')
    with zipfile.ZipFile(CALVIN_ZIP, 'r') as zf:
        all_names = set(zf.namelist())
        print(f'Files in zip: {len(all_names):,}')

        for split, max_ep in [('training', TRAIN_EP), ('validation', VAL_EP)]:
            local = f'{CALVIN_PATH}/{split}'
            os.makedirs(f'{local}/lang_annotations', exist_ok=True)

            ann_zip  = f'task_D_D/{split}/lang_annotations/auto_lang_ann.npy'
            ann_local = f'{local}/lang_annotations/auto_lang_ann.npy'
            if not os.path.exists(ann_local):
                zf.extract(ann_zip, '/content/calvin_data')

            indx, _ = load_ann(split)
            needed  = needed_indices(indx, max_ep)
            missing = [i for i in needed
                       if not os.path.exists(f'{local}/episode_{i:07d}.npz')]
            print(f'[{split}] {len(needed)} needed, {len(missing)} to extract')

            for i in tqdm(missing, desc=split):
                zname = f'task_D_D/{split}/episode_{i:07d}.npz'
                if zname in all_names:
                    zf.extract(zname, '/content/calvin_data')

# ─── Strategy B: download individual files ────────────────────────────────────
elif STRATEGY == 'individual_files':
    for split, max_ep in [('training', TRAIN_EP), ('validation', VAL_EP)]:
        local = f'{CALVIN_PATH}/{split}'
        cache = f'{DRIVE_CACHE}/{split}'
        os.makedirs(f'{local}/lang_annotations', exist_ok=True)
        os.makedirs(f'{cache}/lang_annotations', exist_ok=True)

        # Annotation file
        ann_local = f'{local}/lang_annotations/auto_lang_ann.npy'
        ann_cache = f'{cache}/lang_annotations/auto_lang_ann.npy'
        if not os.path.exists(ann_local):
            if os.path.exists(ann_cache):
                shutil.copy2(ann_cache, ann_local)
            else:
                urllib.request.urlretrieve(
                    f'{CALVIN_BASE}/{split}/lang_annotations/auto_lang_ann.npy',
                    ann_local)
                shutil.copy2(ann_local, ann_cache)

        indx, _ = load_ann(split)
        needed  = needed_indices(indx, max_ep)
        missing = [i for i in needed
                   if not os.path.exists(f'{local}/episode_{i:07d}.npz')]
        print(f'[{split}] {len(needed)} NPZs needed, {len(missing)} to download')

        failed = 0
        for i in tqdm(missing, desc=split):
            fname  = f'episode_{i:07d}.npz'
            lpath  = f'{local}/{fname}'
            cpath  = f'{cache}/{fname}'
            if os.path.exists(cpath):
                shutil.copy2(cpath, lpath)
            else:
                try:
                    urllib.request.urlretrieve(f'{CALVIN_BASE}/{split}/{fname}', lpath)
                    shutil.copy2(lpath, cpath)
                except:
                    failed += 1
        print(f'[{split}] done, {failed} failed')

# ─── Strategy C: need full zip ────────────────────────────────────────────────
elif STRATEGY == 'need_full_zip':
    !df -h /content
    print('\nSee disk space above.')
    print('If /content has >165 GB free, run this to download zip to local disk:')
    print()
    print('  !apt-get install -q -y aria2')
    print(f'  !aria2c -x 16 -s 16 --file-allocation=none \\')
    print(f'      -o task_D_D.zip -d /content \\')
    print(f'      http://calvin.cs.uni-freiburg.de/dataset/task_D_D.zip')
    print()
    print('Then re-run this cell — it will detect the complete zip and extract selectively.')
    raise RuntimeError('See instructions above — manual step required')

# ── Verify ────────────────────────────────────────────────────────────────────
train_npz = list(Path(f'{CALVIN_PATH}/training').glob('episode_*.npz'))
val_npz   = list(Path(f'{CALVIN_PATH}/validation').glob('episode_*.npz'))
total_gb  = sum(f.stat().st_size for f in train_npz + val_npz) / 1e9
print(f'\nLocal disk — training: {len(train_npz):,}  validation: {len(val_npz):,}  ({total_gb:.2f} GB)')
assert len(train_npz) > 100, 'Too few training files — something went wrong'
print('CALVIN_PATH:', CALVIN_PATH)
print('Done ✓')

FileNotFoundError: [Errno 2] No such file or directory: '/content/calvin_data/task_D_D/training/lang_annotations/auto_lang_ann.npy'

In [ ]:
# ── Cell 6: Smoke test — verify loader works before full run ─────────────────
import sys
sys.path.insert(0, REPO_DIR)

from data.trajectory_dataset import load_calvin

print('Loading a few CALVIN episodes to verify loader...')
eps = load_calvin(CALVIN_PATH, split='training')

print(f'Episodes loaded: {len(eps)}')
print(f'Keys: {list(eps[0].keys())}')
print(f'Frames shape:  {eps[0]["frames"].shape}')
print(f'Actions shape: {eps[0]["actions"].shape}  dtype={eps[0]["actions"].dtype}')
print(f'Action range:  {eps[0]["actions"].min()} – {eps[0]["actions"].max()}  (expect 0–13)')
print(f'Action vectors:{eps[0]["action_vectors"].shape}  (expect T×7)')
print(f'Instruction:   {eps[0]["instruction"]}')
print('Loader OK')

In [ ]:
# ── Cell 7: Configure run ─────────────────────────────────────────────────────
# ↓ ONLY change these two values ↓

START_FROM = 0    # Resume from ablation index if session died (0 = start fresh)
                  # Indices: 0=full_vera  1=bc_baseline  2=no_exp  3=no_act
                  #          4=no_lang    5=no_history_tf 6=no_reward_gate
                  #          7=no_dual_head  8=corrupted_conseq

DRY_RUN = False   # Set True first to verify (2 epochs, synthetic data, ~2 min)
                  # Set False for the real run

# ─────────────────────────────────────────────────────────────────────────────
# Point checkpoints to Drive so they survive session crashes
import yaml, os

cfg_path = f'{REPO_DIR}/configs/calvin_config.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

# Redirect output to Drive
cfg['data']['episodes_path']   = CALVIN_PATH
cfg['training']['output_dir']  = f'{DRIVE_ROOT}/checkpoints'
cfg['training']['device']      = 'cuda'
cfg['training']['num_workers'] = 2    # Colab works better with 2

# Save patched config
colab_cfg_path = f'{REPO_DIR}/configs/calvin_colab_runtime.yaml'
with open(colab_cfg_path, 'w') as f:
    yaml.dump(cfg, f)

print('Config ready')
print(f'  Dataset:  {CALVIN_PATH}')
print(f'  Output:   {DRIVE_ROOT}/checkpoints')
print(f'  Epochs:   {cfg["training"]["epochs"]} (max, early stopping patience=10)')
print(f'  Ablations to run: {9 - START_FROM} (starting from index {START_FROM})')
print(f'  Seeds: 42, 123, 456')
print(f'  DRY RUN: {DRY_RUN}')

In [ ]:
# ── Cell 8: Run ablations ─────────────────────────────────────────────────────
# This cell runs all 9 ablations × 3 seeds sequentially.
# Checkpoints save to Drive after every epoch — safe to interrupt and resume.
# Expected time on T4:  ~20-40 min per (ablation × seed)  →  ~9-18 hours total
# Expected time on A100: ~8-15 min per (ablation × seed)  →  ~4-7  hours total

import subprocess, sys

cmd = [
    sys.executable,
    f'{REPO_DIR}/scripts/run_calvin_ablations.py',
    '--calvin_path', CALVIN_PATH,
    '--config',      colab_cfg_path,
    '--out',         f'{DRIVE_ROOT}/checkpoints',
    '--start_from',  str(START_FROM),
]
if DRY_RUN:
    cmd.append('--dry_run')

print('Running:', ' '.join(cmd))
print('='*65)

# Run with live output
result = subprocess.run(cmd, cwd=REPO_DIR)
print('='*65)
print('Exit code:', result.returncode)

In [ ]:
# ── Cell 9: View results ──────────────────────────────────────────────────────
import json, numpy as np

results_path = f'{DRIVE_ROOT}/checkpoints/calvin_ablation_summary.json'

with open(results_path) as f:
    results = json.load(f)

print(f'  {"Ablation":<46} {"Mean":>7}  {"Std":>6}  Seeds')
print(f'  {"-"*46} {"-"*7}  {"-"*6}  {"-"*22}')
for slug, v in results.items():
    seeds = '  '.join(f'{x:.4f}' for x in v['seeds'].values())
    print(f'  {v["display"]:<46} {v["mean_val_acc"]:.4f}  {v["std_val_acc"]:.4f}  {seeds}')

In [ ]:
# ── Cell 10: Download results JSON to local machine ───────────────────────────
from google.colab import files
import shutil, os

# Bundle the summary JSON + all training logs into one zip
zip_path = '/content/calvin_results.zip'
shutil.make_archive(
    '/content/calvin_results', 'zip',
    f'{DRIVE_ROOT}/checkpoints'
)
files.download(zip_path)
print('Download started')